# Capítulo VIII: Pruebas, Despliegue y Mejoras del Modelo

En esta fase final, el modelo entrenado se pondrá en producción simulada. 
1. **Despliegue con Gradio:** Creación de una interfaz interactiva para inferencia en tiempo real.
2. **Subida a Hugging Face:** Versionado del modelo en la nube.
3. **Comparación (Benchmarking):** Demostraremos matemáticamente que el uso de Sentimiento (NLP) mejora la predicción frente a un modelo puramente numérico.
4. **Fine-Tuning:** Optimización de hiperparámetros para mejorar el error cuadrático medio (RMSE).

In [1]:
import gradio as gr
import xgboost as xgb
import pandas as pd
import numpy as np

# 1. Cargar el modelo que guardamos en el Hito anterior
modelo_cargado = xgb.XGBRegressor()
modelo_cargado.load_model("xgboost_hedgemind_model.json")

# 2. Función de Predicción en Tiempo Real
def predecir_rendimiento(close, volume, rsi, sentiment, news_vol):
    # Crear un DataFrame con el mismo formato que el entrenamiento
    input_data = pd.DataFrame([[close, volume, rsi, sentiment, news_vol]], 
                            columns=['close_price', 'trading_volume', 'rsi_14', 'sentiment_score', 'news_volume'])
    
    prediccion = modelo_cargado.predict(input_data)[0]
    
    # Formatear la salida para que sea legible
    tendencia = "📈 ALCISTA" if prediccion > 0 else "📉 BAJISTA"
    porcentaje = prediccion * 100
    
    return f"{tendencia} | Variación esperada: {porcentaje:.2f}%"

# 3. Diseño de la Interfaz Web
interfaz = gr.Interface(
    fn=predecir_rendimiento,
    inputs=[
        gr.Number(label="Precio de Cierre (Close Price)", value=120.50),
        gr.Number(label="Volumen de Transacciones (M)", value=50000000),
        gr.Slider(minimum=0, maximum=100, label="Indicador RSI (14 días)", value=55),
        gr.Slider(minimum=-1, maximum=1, step=0.01, label="Sentiment Score (VADER/FinBERT)", value=0.45),
        gr.Number(label="Volumen de Noticias (Día)", value=25)
    ],
    outputs=gr.Textbox(label="Predicción del HedgeMind-NVDA (Target Return)"),
    title="🧠 HedgeMind-NVDA: Terminal de Inferencia Híbrida",
    description="Introduce los datos técnicos y el análisis de sentimiento del día para predecir el movimiento de las acciones de NVIDIA para la jornada de mañana."
)

# Lanzar la aplicación dentro del Notebook (crea también un link público temporal de 72h)
interfaz.launch(share=True, inline=True)

c:\Users\faust\miniconda3\envs\tfm_ai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


Interfas de Gradio


In [2]:
import gradio as gr
import xgboost as xgb
import pandas as pd
import numpy as np

# 1. Cargar el modelo que guardamos en el Hito anterior
modelo_cargado = xgb.XGBRegressor()
modelo_cargado.load_model("xgboost_hedgemind_model.json")

# 2. Función de Predicción en Tiempo Real
def predecir_rendimiento(close, volume, rsi, sentiment, news_vol):
    # Crear un DataFrame con el mismo formato que el entrenamiento
    input_data = pd.DataFrame([[close, volume, rsi, sentiment, news_vol]], 
                              columns=['close_price', 'trading_volume', 'rsi_14', 'sentiment_score', 'news_volume'])
    
    prediccion = modelo_cargado.predict(input_data)[0]
    
    # Formatear la salida para que sea legible
    tendencia = "📈 ALCISTA" if prediccion > 0 else "📉 BAJISTA"
    porcentaje = prediccion * 100
    
    return f"{tendencia} | Variación esperada: {porcentaje:.2f}%"

# 3. Diseño de la Interfaz Web
interfaz = gr.Interface(
    fn=predecir_rendimiento,
    inputs=[
        gr.Number(label="Precio de Cierre (Close Price)", value=120.50),
        gr.Number(label="Volumen de Transacciones (M)", value=50000000),
        gr.Slider(minimum=0, maximum=100, label="Indicador RSI (14 días)", value=55),
        gr.Slider(minimum=-1, maximum=1, step=0.01, label="Sentiment Score (VADER/FinBERT)", value=0.45),
        gr.Number(label="Volumen de Noticias (Día)", value=25)
    ],
    outputs=gr.Textbox(label="Predicción del HedgeMind-NVDA (Target Return)"),
    title="🧠 HedgeMind-NVDA: Terminal de Inferencia Híbrida",
    description="Introduce los datos técnicos y el análisis de sentimiento del día para predecir el movimiento de las acciones de NVIDIA para la jornada de mañana."
)

# Lanzar la aplicación en red local (instantáneo y sin bloqueos)
interfaz.launch(share=False, inline=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Subida a huggin Face Hub

In [15]:
from huggingface_hub import HfApi, login
import os
from dotenv import load_dotenv


load_dotenv() 

print("🚀 Preparando subida a Hugging Face Hub...")

token_hf = os.getenv("HF_TOKEN")
repo_id = os.getenv("HF_MODEL")

# 1. Validación Estricta de Variables de Entorno
if not token_hf:
    print("❌ ERROR: No se encontró HF_TOKEN en el archivo .env")
elif not repo_id:
    print("❌ ERROR: No se encontró HF_MODEL en el archivo .env")
else:
    print(f"✅ Credenciales cargadas. Intentando subir al repositorio: {repo_id}")
    try:
        # 2. Login con el token
        login(token=token_hf)
        api = HfApi()
        
        # 3. Creación/Verificación del Repositorio
        api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
        print("✅ Repositorio verificado.")
        
        # 4. Ajuste de la ruta del modelo (¡Crucial!)
        # Como estamos en la carpeta notebooks/, el modelo probablemente esté en la carpeta superior o en models/
        ruta_modelo = "../hitos/xgboost_hedgemind_model.json" # Ajusta esto si tu modelo está en otra carpeta
        
        if not os.path.exists(ruta_modelo):
            print(f"❌ ERROR: No se encuentra el archivo del modelo en la ruta: {ruta_modelo}")
        else:
            # 5. Subida del Archivo
            api.upload_file(
                path_or_fileobj=ruta_modelo,
                path_in_repo="xgboost_hedgemind_model.json", # Nombre que tendrá en la nube
                repo_id=repo_id,
                repo_type="model",
            )
            print(f"🎉 ¡Modelo desplegado con éxito en la nube!")
            print(f"🔗 Enlace público: https://huggingface.co/{repo_id}")
            
    except Exception as e:
        print(f"⚠️ Error crítico en la comunicación con Hugging Face: {e}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


🚀 Preparando subida a Hugging Face Hub...
✅ Credenciales cargadas. Intentando subir al repositorio: faujarbun/HedgeMind-NVDA
✅ Repositorio verificado.


No files have been modified since last commit. Skipping to prevent empty commit.


🎉 ¡Modelo desplegado con éxito en la nube!
🔗 Enlace público: https://huggingface.co/faujarbun/HedgeMind-NVDA


In [17]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
import xgboost as xgb
import os

print("🚀 INICIANDO PIPELINE DE MACHINE LEARNING (HITO 3)...")

# ==========================================
# 1. CARGA Y PREPARACIÓN DE DATOS
# ==========================================
archivo_local = "../datasets/dataset_entrenamiento_final.csv"
if not os.path.exists(archivo_local):
    raise FileNotFoundError(f"No se encuentra {archivo_local}. Asegúrate de haber descargado los datos de AWS.")

df = pd.read_csv(archivo_local)
df['fecha'] = pd.to_datetime(df['fecha'])
df = df.sort_values('fecha')

# Forzar formato numérico y crear la variable objetivo (Target)
columnas = ['close_price', 'trading_volume', 'rsi_14', 'sentiment_score', 'news_volume']
for col in columnas:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['target_return'] = df['close_price'].pct_change().shift(-1)
df.dropna(inplace=True)

# División Temporal (Train/Test)
split_idx = int(len(df) * 0.8)
X_train = df[columnas].iloc[:split_idx]
X_test = df[columnas].iloc[split_idx:]
y_train = df['target_return'].iloc[:split_idx]
y_test = df['target_return'].iloc[split_idx:]

print(f"✅ Datos listos: {len(X_train)} filas para entrenar, {len(X_test)} para validar.")

# ==========================================
# 2. ENTRENAMIENTO: MODELO HÍBRIDO (TFM)
# ==========================================
print("\n🧠 Entrenando Modelo HedgeMind (Análisis Técnico + Sentimiento NLP)...")
modelo_hibrido = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=150, learning_rate=0.05, random_state=42)
modelo_hibrido.fit(X_train, y_train)

predicciones_hibrido = modelo_hibrido.predict(X_test)
rmse_hibrido = np.sqrt(mean_squared_error(y_test, predicciones_hibrido))

# ==========================================
# 3. ENTRENAMIENTO: MODELO BASELINE (CLÁSICO)
# ==========================================
print("⚙️ Entrenando Modelo Clásico Baseline (Solo Análisis Técnico)...")
features_baseline = ['close_price', 'trading_volume', 'rsi_14']
X_train_base = X_train[features_baseline]
X_test_base = X_test[features_baseline]

modelo_base = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=150, learning_rate=0.05, random_state=42)
modelo_base.fit(X_train_base, y_train)

predicciones_base = modelo_base.predict(X_test_base)
rmse_base = np.sqrt(mean_squared_error(y_test, predicciones_base))

# ==========================================
# 4. BENCHMARKING Y CONCLUSIONES
# ==========================================
print("\n" + "="*50)
print("⚖️ RESULTADOS FINALES DEL BENCHMARKING:")
print("="*50)
print(f"📉 RMSE Modelo Baseline (Sin Noticias): {rmse_base:.5f}")
print(f"📈 RMSE Modelo HedgeMind (Con Noticias): {rmse_hibrido:.5f}")

if rmse_hibrido < rmse_base:
    mejora = ((rmse_base - rmse_hibrido) / rmse_base) * 100
    print(f"\n🏆 CONCLUSIÓN: ¡Éxito! El análisis de sentimiento mejoró la precisión en un {mejora:.2f}%")
else:
    print("\n⚠️ El modelo clásico fue igual o mejor. (Esto es normal al inicio: se necesitan meses de recolección en Kafka para que el sentimiento venza al análisis técnico puro).")

🚀 INICIANDO PIPELINE DE MACHINE LEARNING (HITO 3)...
✅ Datos listos: 5488 filas para entrenar, 1373 para validar.

🧠 Entrenando Modelo HedgeMind (Análisis Técnico + Sentimiento NLP)...
⚙️ Entrenando Modelo Clásico Baseline (Solo Análisis Técnico)...

⚖️ RESULTADOS FINALES DEL BENCHMARKING:
📉 RMSE Modelo Baseline (Sin Noticias): 0.03488
📈 RMSE Modelo HedgeMind (Con Noticias): 0.03488

⚠️ El modelo clásico fue igual o mejor. (Esto es normal al inicio: se necesitan meses de recolección en Kafka para que el sentimiento venza al análisis técnico puro).
